# Scoring LLMs with `scorio.eval`

We use [Scorio](https://github.com/mohsenhariri/scorio) for scoring, ranking and
aggregating stochastic model responses. This notebook covers `scorio.eval`, which turns
a matrix of outcomes into a score with an uncertainty attached to it.

Every function in the module takes the same first argument, a matrix `R` of shape
`M x N`, one row per question and one column per sampled trial. That is exactly the
shape of a [Scorio Trace](https://huggingface.co/datasets/harimo/scorio-trace) split, 30 questions by 80 seeds.

|  |  |
| --- | --- |
| module | [`scorio/eval`](https://github.com/mohsenhariri/scorio/tree/main/scorio/eval) |
| method reference | [`scorio/eval/README.md`](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/README.md) |
| paper | [Don't Pass@k: A Bayesian Framework for Large Language Model Evaluation](https://openreview.net/forum?id=PTXi3Ef4sT), ICLR 2026 ([arXiv](https://arxiv.org/abs/2510.04265)) |
| video | [walkthrough](https://github.com/user-attachments/assets/7cb72b44-7e24-40f5-b198-4c102fe2d184) |
| docs | [scorio.readthedocs.io/en/latest/api/eval](https://scorio.readthedocs.io/en/latest/api/eval.html) |
| install | `pip install scorio` |

The data comes from the [Scorio Trace](https://huggingface.co/datasets/harimo/scorio-trace) dataset, see [trace.ipynb](https://github.com/mohsenhariri/scorio/blob/main/notebooks/datasets/trace/trace.ipynb).

In [1]:
import pandas as pd
from datasets import load_dataset
from scorio import eval

repo_name = "harimo/scorio-trace"
task = "aime25"

rows = (load_dataset(repo_name, "meta", split=task)
        .select_columns(["model_key", "data_id", "seed", "is_correct"])
        .to_pandas())


def outcomes(model_key):
    """R for one model, 30 questions x 80 seeds, entries in {0, 1}."""
    one = rows[rows.model_key == model_key].sort_values(["data_id", "seed"])
    return one.is_correct.to_numpy().astype(int).reshape(30, 80)


R = outcomes("Phi-4-reasoning")
print(R.shape, R.dtype)
print(R[:3, :12])

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

(30, 80) int64
[[1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 0 1 1 0 0 0 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1]]


## Bayes@N

`bayes_ci` returns the [Bayes@N](https://arxiv.org/abs/2510.04265) posterior
score, its standard deviation and a 95% credible interval as `(mu, sigma, lo, hi)`.

In [2]:
mu, sigma, lo, hi = eval.bayes_ci(R)
print(f"Bayes@N  {mu:.3f} +- {sigma:.3f}   95% interval [{lo:.3f}, {hi:.3f}]")

Bayes@N  0.598 +- 0.007   95% interval [0.584, 0.613]


## Pass@k, Maj@k and the rest

The same `R` answers several different questions, and the answers are far apart. At 8
samples this model solves at least one question out of eight 87% of the time, wins a
strict majority 59% of the time, and gets all eight right only 26% of the time.

`pass_at_k` is the usual unbiased estimator, `maj_at_k` is strict majority,
`pass_hat_k` is all-of-k (also written Pass^k or `unanimous_at_k`), and `auc_at_k` is
the area under Pass@1 through Pass@k.

In [3]:
ks = [1, 2, 4, 8, 16, 80]
table = pd.DataFrame({
    "pass@k": [eval.pass_at_k(R, k) for k in ks],
    "maj@k": [eval.maj_at_k(R, k) for k in ks],
    "pass^k": [eval.pass_hat_k(R, k) for k in ks],
    "auc@k": [eval.auc_at_k(R, k) for k in ks],
}, index=pd.Index(ks, name="k"))

display(table.round(3))

,pass@k,maj@k,pass^k,auc@k
k,,,,
1,0.601,0.601,0.601,0.601
2,0.735,0.467,0.467,0.668
4,0.822,0.536,0.342,0.746
8,0.867,0.590,0.256,0.805
16,0.893,0.627,0.211,0.846
80,0.933,0.667,0.167,0.908


## What more samples buy you

Slicing columns off `R` is a budget sweep. The score barely moves, the interval shrinks
by a factor of six. One sample per question and 80 samples per question give you the same
answer, but only the second one lets you say it out loud.

In [4]:
sweep = pd.DataFrame(
    [eval.bayes_ci(R[:, :n]) for n in (1, 2, 4, 8, 16, 32, 80)],
    columns=["mu", "sigma", "lo", "hi"],
    index=pd.Index([1, 2, 4, 8, 16, 32, 80], name="samples"),
)
sweep["width"] = sweep.hi - sweep.lo
display(sweep.round(3))

,mu,sigma,lo,hi,width
samples,,,,,
1,0.567,0.043,0.482,0.651,0.169
2,0.600,0.037,0.528,0.672,0.145
4,0.594,0.029,0.537,0.652,0.114
8,0.590,0.022,0.547,0.633,0.086
16,0.596,0.016,0.565,0.627,0.062
32,0.603,0.012,0.580,0.626,0.045
80,0.598,0.007,0.584,0.613,0.029


## Models that look different and are not

Five models on this task, ordered by accuracy. The top four are within 0.015 of each
other and their intervals overlap, so the leaderboard order between them is noise. The
fifth one is separated.

This is the argument of the paper. A ranking without intervals invites you to read
differences that the data does not support.

In [5]:
models = ["Phi-4-reasoning", "OpenThinker2-32B", "Light-R1-14B-DS",
          "FuseO1-DeepSeekR1-QwQ-SkyT1-Flash-32B-Preview", "NVIDIA-Nemotron-Nano-9B-v2"]

summary = pd.DataFrame([eval.bayes_ci(outcomes(m)) for m in models],
                       columns=["mu", "sigma", "lo", "hi"], index=models)

display(summary.round(3))

,mu,sigma,lo,hi
Phi-4-reasoning,0.598,0.007,0.584,0.613
OpenThinker2-32B,0.593,0.007,0.580,0.606
Light-R1-14B-DS,0.587,0.007,0.573,0.600
FuseO1-DeepSeekR1-QwQ-SkyT1-Flash-32B-Preview,0.583,0.006,0.570,0.595
NVIDIA-Nemotron-Nano-9B-v2,0.547,0.007,0.534,0.560


## Other APIs

`g_pass_at_k_tau` for a tunable success threshold, `mg_pass_at_k` for its average over
thresholds, `max_at_k` for expected best reward, and the geometric and spectrum family.
Every point estimator has an `_ci` companion. Categorical outcomes work too, pass a
weight vector `w` of length `C+1` to `bayes`, `avg` or `max_at_k`.

The full table with references for each method is in
[`scorio/eval/README.md`](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/README.md).
